# einops-rearrange — worked example 1: Merge a time axis into the batch axis (composition)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange

t.manual_seed(0)
np.random.seed(0)

## Concept

`rearrange` can **compose** two adjacent axes into one by parenthesizing them on the output side: `'b t c -> (b t) c'`. The composed axis iterates in row-major order, so the leftmost named axis (`b`) varies slowest and the rightmost (`t`) varies fastest. This is the standard trick for turning a `(batch, time, feat)` sequence batch into a flat `(batch*time, feat)` matrix that a per-step MLP can consume.

## Worked solution

We have a tensor of shape `(b, t, c)` — `b` independent sequences, each `t` timesteps long, each timestep a `c`-vector. We want one big matrix of `(b*t, c)` rows so a plain `nn.Linear` can process every timestep at once.

**Step 1 — name every input axis.** The left side of the pattern is `b t c`. Three identifiers for the three input dims; nothing is dropped.

**Step 2 — decide the output layout.** We want a 2-D result. The feature axis `c` stays on its own. The other two axes must merge, so we write `(b t)` — the parentheses mean *compose these into one axis*.

**Step 3 — why the order inside the parens matters.** `(b t)` lays out the rows so that all `t` rows of sequence 0 come first, then all `t` rows of sequence 1, and so on — `b` slowest, `t` fastest. Writing `(t b)` instead would interleave the sequences, which is *not* what a row-major flatten means.

**Step 4 — verify the count.** Input has `b*t*c` elements; output `(b*t, c)` also has `b*t*c`. einops checks this for us and raises if it ever fails, so a wrong pattern is caught loudly rather than silently corrupting data.

In [ ]:
def merge_time_into_batch(x: Tensor) -> Tensor:
    return rearrange(x, 'b t c -> (b t) c')


np.random.seed(0)
t.manual_seed(0)
x = t.arange(2 * 3 * 4).reshape(2, 3, 4)
out = merge_time_into_batch(x)
print('input shape :', tuple(x.shape))
print('output shape:', tuple(out.shape))
print('row 0 of seq0 == input[0,0]?', bool(t.equal(out[0], x[0, 0])))
print('row 3 (= seq1 t0) == input[1,0]?', bool(t.equal(out[3], x[1, 0])))